# Experiment

In [17]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path

In [18]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

## Index Creation/Loading

In [20]:
index_dir_path = Path("index/trec_covid_index").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [21]:
os.makedirs(index_dir, exist_ok=True)

In [22]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))
        
        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [23]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index not found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\src\index\trec_covid_index. Building it now (this may take a few minutes)...


cord19/trec-covid documents:   0%|          | 0/192509 [00:00<?, ?it/s]

13:59:11.820 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
14:00:11.822 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:377)
14:00:16.829 [main] WA

## Different Query Verbosity Levels

In [26]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [27]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Models

In [33]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
ql_dir = pt.terrier.Retriever(index_ref, wmodel="DirichletLM")
ql_jm = pt.terrier.Retriever(index_ref, wmodel="Hiemstra_LM")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

In [34]:
models = [bm25, ql_dir, ql_jm, rm3_pipe]
model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3"]

## Experiment

In [35]:
eval_metrics = ["map", "ndcg_cut_10"]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [36]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.601847,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.444853,14.0,36.0,3.445424e-04,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.389103,2.0,48.0,2.471805e-09,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.580049,30.0,20.0,1.205768e-01,21.0,25.0,2.834572e-01




Results for DESCRIPTION Queries:


,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.623256,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.505937,9.0,41.0,2.067988e-05,15.0,35.0,0.000169
2,QL (JM),0.125899,0.444361,1.0,49.0,1.163376e-11,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.655691,32.0,18.0,4.463815e-02,26.0,20.0,0.126493




Results for NARRATIVE Queries:


,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.518681,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.347096,8.0,42.0,4.518420e-07,11.0,35.0,0.000005
2,QL (JM),0.117736,0.388177,7.0,43.0,1.239431e-05,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.489098,19.0,31.0,1.503040e-01,17.0,27.0,0.135629


## Experiment with Dense Retrieval (DPR)

In [39]:
!pip install -q torch torchvision torchaudio transformers
!pip install -q faiss-cpu 
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_dr.git


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

dense_index_dir = str(Path("index/trec_covid_dense").resolve())

if os.path.exists(os.path.join(dense_index_dir, "pt_meta.json")):
    print("Found downloaded dense index! Loading...")
    dpr_model = ptdr.SBertBiEncoder('sentence-transformers/all-MiniLM-L6-v2')
    dense_index = ptdr.FlexIndex(dense_index_dir)
    dpr_pipe = dpr_model >> dense_index
    
    print("DPR Pipeline ready!")
else:
    print(f"Could not find the dense index at {dense_index_dir}. Please check where you extracted it!")

Found downloaded dense index! Loading...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\zosia\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zosia\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

DPR Pipeline ready!


In [41]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR (all-MiniLM)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.601847,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.444853,14.0,36.0,3.445424e-04,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.389103,2.0,48.0,2.471805e-09,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.580049,30.0,20.0,1.205768e-01,21.0,25.0,2.834572e-01
4,DPR (all-MiniLM),0.075150,0.300213,3.0,47.0,2.424788e-09,6.0,43.0,2.638420e-09




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.623256,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.505937,9.0,41.0,2.067988e-05,15.0,35.0,0.000169
2,QL (JM),0.125899,0.444361,1.0,49.0,1.163376e-11,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.655691,32.0,18.0,4.463815e-02,26.0,20.0,0.126493
4,DPR (all-MiniLM),0.119925,0.448695,6.0,44.0,2.219091e-09,12.0,38.0,0.000475




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,ndcg_cut_10,map +,map -,map p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.518681,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.347096,8.0,42.0,4.518420e-07,11.0,35.0,0.000005
2,QL (JM),0.117736,0.388177,7.0,43.0,1.239431e-05,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.489098,19.0,31.0,1.503040e-01,17.0,27.0,0.135629
4,DPR (all-MiniLM),0.111516,0.367265,14.0,36.0,2.399987e-03,11.0,35.0,0.000372
